In [0]:
df = spark.read.csv('/Workspace/Users/filipedbrito94@gmail.com/serasa-case/serasa_case.csv', header=True, inferSchema=True, sep=';')

# Ajuste nome de coluna
df = df.withColumnRenamed("convercao_efetiva", "conversao_efetiva")

# Transformando df em view para trabalhar em sql
df.createOrReplaceTempView("credit_data")

In [0]:
%sql
-- df como view

In [0]:
%sql
select
  *
from
  credit_data
limit 10

data,fx_score_de_credito,fx_de_renda,Negativado,faixa_etaria,total_users_simulando,creditType,usuario_elegivel,possui_oferta,convercao_efetiva,receita_gerada
2025-02-03,301-400,0-1600,true,0-20,490,digital-account,360.0,359.0,17.0,19.5
2025-02-03,301-400,0-1600,true,0-20,490,personal-loan-fgts,20.0,11.0,5.0,6.0
2025-02-03,301-400,0-1600,true,0-20,490,personal-loan,66.0,43.0,10.0,0.0
2025-02-03,301-400,0-1600,true,0-20,490,vehicle-guarantee-loan,2.0,0.0,0.0,0.0
2025-02-14,701-800,4001-5000,false,31-40,7196,personal-loan-fgts,40.0,2.0,0.0,0.0
2025-02-14,701-800,4001-5000,false,31-40,7196,digital-account,1408.0,1070.0,7.0,7.5
2025-02-14,701-800,4001-5000,false,31-40,7196,personal-loan,356.0,215.0,13.0,24.0
2025-02-14,701-800,4001-5000,false,31-40,7196,credit-card,7842.0,3714.0,66.0,604.5
2025-02-14,701-800,4001-5000,false,31-40,7196,vehicle-guarantee-loan,53.0,25.0,4.0,36.0
2025-02-09,701-800,1601-2000,false,41-50,1531,personal-loan-fgts,12.0,3.0,0.0,0.0


In [0]:
%sql
select 
  creditType,
  SUM(receita_gerada) as receita_total
from credit_data
group by 1
order by 2 desc 

creditType,receita_total
credit-card,1399411.5
personal-loan,713703.0
digital-account,326091.0
vehicle-guarantee-loan,208630.5
personal-loan-fgts,57268.5
home-equity,15867.0
payroll-loan,8830.5
null,0.0


In [0]:
df.summary().display()

summary,fx_score_de_credito,fx_de_renda,faixa_etaria,total_users_simulando,creditType,usuario_elegivel,possui_oferta,convercao_efetiva,receita_gerada
count,107821,107821,107821,107821,105651,107821,107821,107821,107821
mean,null,null,null,1570.1922352788417,null,361.0423479656097,168.17392715704733,6.510410773411488,25.31790653026776
stddev,null,null,null,4074.874114262147,null,1863.148353415742,649.5059352659267,28.562346188169364,103.40508863211883
min,0-100,0-1600,0-20,1,credit-card,0.0,0.0,0.0,0.0
25%,null,null,null,62,null,2.0,1.0,0.0,0.0
50%,null,null,null,371,null,14.0,8.0,0.0,0.0
75%,null,null,null,1308,null,110.0,63.0,3.0,7.5
max,901-1000,9001+,80+,59394,vehicle-guarantee-loan,63865.0,14003.0,853.0,3087.0


Considerações do dataset:
- Volume de registros bate com a planilha recebida no email. Importação ok.
- Poucos casos nulos preenchidos, apenas pra credittype

In [0]:
%sql
select
  distinct 
  fx_score_de_credito,
  fx_de_renda,
  negativado,
  faixa_etaria
from 
  credit_data

fx_score_de_credito,fx_de_renda,negativado,faixa_etaria
401-500,6001-7000,true,41-50
501-600,6001-7000,false,0-20
601-700,6001-7000,false,71-80
101-200,8001-9000,true,21-30
401-500,3001-4000,false,21-30
701-800,2001-3000,false,71-80
401-500,1601-2000,true,71-80
301-400,8001-9000,false,41-50
101-200,4001-5000,false,41-50
0-100,9001+,true,41-50


Dicionário
  - **Data**: data de registro
  - **fx_score_de_credito**: faixas de score que determinam o perfil de crédito do consumidor. Quanto maior, melhor.
  - **fx_de_renda**: faixa de renda estimada do consumidor.
  - **Negativado**: Consumidor possui ou não divida negativada.
  - **faixa_etaria**: idade consumidor.
  - **total_usuarios_simulando**: volume de consumidores simulando credito.
  - **creditType**: tipo de crédito
  - **usuarios_elegiveis**: quantidade de consumidores elegiveis ao produto de crédito.
  - **possui_oferta**: quantidade de consumidores que tiveram retorno de oferta.
  - **conversao_efetiva**: quantidade de consumidores que converteram a oferta em pedido.
  - **receita_gerada**: receita gerada pelos pedidos efetuados.


In [0]:
%sql
select
  data,
  sum(total_users_simulando) as total_simulando,
  sum(usuario_elegivel) as total_elegiveis,
  sum(possui_oferta) as total_retorno_oferta,
  sum(conversao_efetiva) as total_conversao,
  sum(receita_gerada) as total_receita,
  sum(usuario_elegivel)/sum(total_users_simulando) as taxa_elegibilidade,
  sum(possui_oferta)/sum(usuario_elegivel) as taxa_oferta,
  sum(receita_gerada)/sum(conversao_efetiva) as receita_por_conversao
from 
  credit_data
group by 
  1
order by 
  1

data,total_simulando,total_elegiveis,total_retorno_oferta,total_conversao,total_receita,taxa_elegibilidade,taxa_oferta,receita_por_conversao
2025-02-01,4289358,984057.0,404101.0,16546.0,62029.5,0.22941824860503598,0.4106479604331863,3.7489121237761394
2025-02-02,4086309,947364.0,386211.0,16284.0,62938.5,0.23183856140101006,0.4076690691223226,3.865051584377303
2025-02-03,6013803,1373858.0,600705.0,22098.0,91908.0,0.22845078230863233,0.4372395109247098,4.159109421667119
2025-02-04,6681052,1538596.0,685752.0,23732.0,100461.0,0.23029247489766583,0.44569984583347416,4.233145120512388
2025-02-05,6838099,1546667.0,695991.0,23356.0,103572.0,0.22618376832508566,0.4499940840529991,4.4344922075697895
2025-02-06,6921596,1556440.0,696337.0,23603.0,97050.0,0.2248672127064336,0.4473908406363239,4.111765453544041
2025-02-07,6072786,1396228.0,626365.0,23237.0,90552.0,0.2299155609962215,0.44861226103473073,3.8968885828635367
2025-02-08,4751810,1099559.0,477381.0,17693.0,76002.0,0.23139793047280932,0.4341567846745832,4.295597128808003
2025-02-09,4274349,981217.0,421131.0,17193.0,66630.0,0.2295594019112618,0.4291925231625624,3.875414412842436
2025-02-10,6604042,1513003.0,669691.0,24626.0,107607.0,0.22910257081950722,0.4426237092722222,4.369649963453261


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# Possíveis perfis

In [0]:
%sql
 -- score de crédito
select fx_score_de_credito, count(*) 
from credit_data
group by 1
order by 2 desc

fx_score_de_credito,count(*)
501-600,16084
401-500,15833
301-400,14453
601-700,11169
701-800,10418
201-300,10329
801-900,9285
901-1000,8622
101-200,8095
0-100,3533


In [0]:
%sql
 -- negativado
select negativado, count(*) 
from credit_data
group by 1

negativado,count(*)
true,36058
false,71763


In [0]:
%sql
 -- faixa etária 
select faixa_etaria, count(*) 
from credit_data
group by 1
order by 2 desc

faixa_etaria,count(*)
31-40,20122
41-50,19526
21-30,19075
51-60,18075
61-70,13463
0-20,9476
71-80,6522
80+,1562


# perfil_cliente — racional

a base tem várias variáveis categóricas (score, renda, negativado, idade) que, combinadas, geram muitas possibilidades de perfis.

isso dificulta:

* análise
* leitura de resultados
* definição de ações (produto, crm, marketing)

a ideia da classificação é:

> reduzir a variabilidade dessas combinações em poucos grupos mais interpretáveis

ou seja:

* agrupar clientes com comportamento esperado semelhante
* facilitar comparação de performance (conversão, receita)
* permitir direcionamento de ação (quem atacar, como abordar)

os agrupamentos são baseados principalmente em:

* risco (score)
* capacidade (renda)
* restrição (negativado)

idade entra como ajuste, não como driver principal
```


In [0]:
%sql

with base as (select
  *,
  case 
    when fx_score_de_credito in ('701-800','801-900','901-1000') then 'alto'
    when fx_score_de_credito in ('501-600','601-700') then 'medio'
    else 'baixo'
  end as score_grupo,
  case
    when fx_de_renda in ('6001-7000','7001-8000','8001-9000','9001+') then 'alta'
    when fx_de_renda in ('3001-4000','4001-5000','5001-6000') then 'media'
    else 'baixa'
  end as renda_grupo,
  case
    when faixa_etaria in ('0-20','21-30') then 'jovem'
    when faixa_etaria in ('31-40','41-50','51-60') then 'adulto'
    else 'idoso'
  end as idade_grupo
from credit_data

)

, perfil as (
select
  data,
  case
    when negativado = true then 'Restrito'
    when score_grupo = 'alto' and renda_grupo = 'alta' then '1. Premium'
    when score_grupo = 'alto' then '2. Bom Pagador'
    when score_grupo = 'medio' and renda_grupo = 'alta' then '3. Potencial (Alta Renda)'
    when score_grupo = 'medio' and renda_grupo = 'media' and idade_grupo = 'adulto' then '4. Medio Consolidado'
    when score_grupo = 'medio' and renda_grupo = 'media' then '5. Medio Jovem'
    when score_grupo = 'medio' and renda_grupo = 'baixa' then '6. Medio Baixa Renda'
    when score_grupo = 'baixo' and renda_grupo = 'baixa' and idade_grupo = 'jovem' then '7. Inicio crédito'
    when score_grupo = 'baixo' and renda_grupo = 'baixa' then '8. Baixa qualidade'
    when score_grupo = 'baixo' then '9. Baixo score'
  end as perfil_cliente,
  case
    when creditType = 'personal-loan' then 'Emprestimo pessoal'
    when creditType = 'vehicle-guarantee-loan' then 'Emprestimo com garantia veiculo'
    when creditType = 'home-equity' then 'Emprestimo com garantia imovel'
    when creditType = 'payroll-loan' then 'Emprestimo consignado'
    when creditType = 'credit-card' then 'Cartao de crédito'
    when creditType = 'digital-account' then 'Conta digital'
    when creditType = 'personal-loan-fgts' then 'Emprestimo fgts'
    else 'Outros'
  end as produto,  
  sum(total_users_simulando) as total_simulando,
  sum(usuario_elegivel) as total_elegiveis,
  sum(possui_oferta) as total_retorno_oferta,
  sum(conversao_efetiva) as total_conversao,
  sum(receita_gerada) as total_receita
from base
group by 1,2,3
)
select
  *,
  case
    when perfil_cliente in ('1. Premium', '2. Bom Pagador') then '1. Alta Qualidade'
    
    when perfil_cliente in (
      '3. Potencial (Alta Renda)',
      '4. Medio Consolidado',
      '5. Medio Jovem'
    ) then '2. Médio Potencial'
    
    when perfil_cliente in (
      '6. Medio Baixa Renda',
      '7. Inicio crédito'
    ) then '3. Baixa Maturidade'
    
    when perfil_cliente in (
      '8. Baixa qualidade',
      '9. Baixo score',
      'Restrito'
    ) then '4. Risco'
    
  end as grupo_cliente

from perfil

data,perfil_cliente,produto,total_simulando,total_elegiveis,total_retorno_oferta,total_conversao,total_receita,grupo_cliente
2025-02-03,8. Baixa qualidade,Emprestimo fgts,71662,435.0,182.0,32.0,37.5,4. Risco
2025-02-04,2. Bom Pagador,Emprestimo pessoal,225843,9884.0,6590.0,446.0,4354.5,1. Alta Qualidade
2025-02-25,6. Medio Baixa Renda,Emprestimo fgts,280378,2797.0,980.0,149.0,175.5,3. Baixa Maturidade
2025-02-11,6. Medio Baixa Renda,Conta digital,251468,45796.0,42822.0,796.0,916.5,3. Baixa Maturidade
2025-02-28,5. Medio Jovem,Conta digital,24115,4644.0,3754.0,54.0,63.0,2. Médio Potencial
2025-02-04,5. Medio Jovem,Conta digital,29221,5261.0,4765.0,42.0,60.0,2. Médio Potencial
2025-02-13,Restrito,Emprestimo com garantia veiculo,165981,562.0,353.0,30.0,1176.0,4. Risco
2025-02-25,1. Premium,Emprestimo com garantia veiculo,73907,520.0,259.0,17.0,328.5,1. Alta Qualidade
2025-02-05,9. Baixo score,Emprestimo com garantia veiculo,74676,721.0,368.0,68.0,1231.5,4. Risco
2025-02-23,1. Premium,Cartao de crédito,42348,45901.0,22627.0,166.0,1999.5,1. Alta Qualidade
